# HIXL 链路管理

Connect 与 Disconnect 是 HIXL 链路创建与断链的核心接口。Connect 建立当前 HIXL engine 到远端 engine 的通信链路，Disconnect 释放链路相关资源。

理解链路管理的关键：链路是传输的前提条件，Connect 之后才能进行 Transfer，Disconnect 之后才能进行 DeregisterMem。

本节学习大纲如下：

- 建链的概念与时机
- Connect 使用方法
- Disconnect 使用方法
- 开发注意事项
- 代码示例


## 1. 建链的概念与时机

### 1.1 建链的概念

HIXL 的远端访问依赖 RDMA/HCCS 等通信通道。建链就是建立当前 engine 到远端 engine 的通信链路，使两端可以进行单边传输。

建链的技术作用：

- 建立 RDMA/HCCS 等通信通道
- 为远端访问准备必要的权限和资源
- 使后续 Transfer 可以直接访问远端地址

### 1.2 Server/Client 角色

Server 和 Client 的区分来自 Initialize 时 local_engine 的写法：

| 角色 | local_engine 格式 | Initialize 行为 |
| --- | --- | --- |
| Server | `host_ip:port` | 启动端口监听，等待远端 Connect |
| Client | `host_ip` | 不启动监听，主动 Connect 到远端 |

角色区分的本质是"谁监听、谁主动连接"。Server 负责监听等待，Client 负责主动发起连接。

### 1.3 建链时机：内存注册后才能建链的原因

一个常见的错误是先 Connect 再 RegisterMem。这会导致远端访问失败，因为建链时 HIXL 才会交换两端注册的内存信息，以便后续进行数据传输。

正确的顺序：Initialize → RegisterMem → Connect。建链前必须完成两端的内存注册:

<img src="./images/connect_timing.png" alt="建链时序图" width="90%">

### 1.4 建链与内存注册的关系

- 建链前需要完成两端内存注册
- 远端地址必须在远端已注册，否则传输会失败
- 建链后注册的内存不支持远端访问

## 2. Connect 使用方法

### 2.1 接口签名

```cpp
Status Connect(const AscendString &remote_engine, int32_t timeout_in_millis = 1000);
```

### 2.2 remote_engine 参数

`remote_engine` 是远端 HIXL 的唯一标识，必须与远端 Initialize 时使用的 `local_engine` 保持一致。

示例：假设两端配置如下

```
local_engine = 10.10.10.1:26000  (Server)
local_engine = 10.10.10.2        (Client)

Connect("10.10.10.1:26000")  // Client 连接到 Server
```

**关键点**：`remote_engine` 是远端 Initialize 时填写的 `local_engine`，两端写入的 IP 和端口不能填反。

### 2.3 timeout_in_millis 参数

`timeout_in_millis` 是建链等待时间，单位 ms，默认值 1000。

- 如果远端 Server 还未启动监听，Client Connect 会等待这个时间后返回 TIMEOUT
- 建议配置 200 ms 以上，网络延迟较大时可适当增加

### 2.4 返回值处理

| 返回值 | 含义 | 处理建议 |
| --- | --- | --- |
| `SUCCESS` | 建链成功 | 进入传输阶段 |
| `PARAM_INVALID` | 参数错误 | 检查 `remote_engine` 格式 |
| `TIMEOUT` | 建链超时 | 检查远端是否启动、网络是否通畅 |
| `ALREADY_CONNECTED` | 重复建链 | 幂等处理，可视为成功继续后续操作 |
| 其他 | 失败 | 打印返回码和 `aclGetRecentErrMsg()` |


## 3. Disconnect 使用方法

### 3.1 接口签名

```cpp
Status Disconnect(const AscendString &remote_engine, int32_t timeout_in_millis = 1000);
```

### 3.2 参数说明

`remote_engine` 与 Connect 时使用的相同，标识要断开的远端 engine。`timeout_in_millis` 是断链等待时间。

### 3.3 释放时机与顺序

Disconnect 必须在 Transfer 完成后调用，在 DeregisterMem 之前。按照依赖关系，传输依赖链路，所以传输结束后才能断链：

`Transfer 完成 → Disconnect → DeregisterMem → aclrtFree → Finalize`

一个常见错误是先 DeregisterMem 再 Disconnect：链路还未释放，内存已解注册，可能导致远端访问异常。

### 3.4 返回值处理

| 返回值 | 含义 | 处理建议 |
| --- | --- | --- |
| `SUCCESS` | 断链成功 | 继续解注册和释放内存 |
| `PARAM_INVALID` | 参数错误 | 检查 `remote_engine` |
| `NOT_CONNECTED` | 未与对端建链 | 链路已不存在或未建链，可继续后续清理 |
| 其他 | 失败 | 打印返回码 |


## 4. 开发注意事项

### 4.1 remote_engine 与远端 local_engine 不匹配

Connect 时填写的 `remote_engine` 必须是远端 Initialize 时使用的 `local_engine`，不能自己随便写一个地址。

错误示例：

- 远端 Initialize 用 `10.10.10.1:26000`
- Connect 时写成 `10.10.10.1`（漏掉端口）
- 结果：返回 PARAM_INVALID

### 4.2 Server/Client 启动时序

Client Connect 前必须确保 Server 已 Initialize 并启动监听。时序颠倒时：

- Client 先启动并 Connect → Server 还未监听 → Client 收到 TIMEOUT
- 建议：Server 先启动 → 等待监听就绪 → Client 再启动

### 4.3 建链后才注册内存

建链时 HIXL 会与对端交换已注册的内存信息。建链后注册的内存不支持远端访问，务必在 Connect 前完成所有本地内存注册。

### 4.4 建链数量上限

最大建链数量为 512，过多建链存在内存 OOM 和性能风险。如果需要大量连接，考虑复用链路或使用连接池。

### 4.5 多线程 context 管理

Connect/Disconnect 要求在 Initialize 同线程调用，或通过：

- `aclrtGetCurrentContext` 获取当前 context
- `aclrtSetCurrentContext` 切换到 Initialize 所在 context

跨线程调用会导致 context 不匹配，接口返回错误。


## 5. 代码示例

在前面的内容 [2.2 HIXL资源管理](./02.02_hixl_resource_management.ipynb) 和 [2.3 HIXL内存管理](./02.03_hixl_memory_management.ipynb) 中，我们已经学习了如何使用资源管理接口（Initilize/Finalize）和内存管理接口（RegisterMem/DeregisterMem），下面将代码示例展示 Connect 与 Disconnect 的使用流程。

```c++
// engine 已完成 Initialize 和 RegisterMem
Status ret = engine.Connect(kRemoteEngine, 3000);
if (ret != SUCCESS) {
    printf("[ERROR] Connect failed, ret = %u\n", ret);
    return -1;
}

// 在 Connect 与 Disconnect 之间执行传输操作
// 具体方法将在下一节介绍

ret = engine.Disconnect(kRemoteEngine, 3000);
if (ret != SUCCESS) {
    printf("[ERROR] Disconnect failed, ret = %u\n", ret);
    return -1;
}

// engine 进行 DeregisterMem 和 Finalize
```

## 课后练习

本节介绍了建链的概念、Connect/Disconnect 的使用方法和开发注意事项，请根据学习内容完成以下题目进行自测。

1. （判断题）Server 端 `local_engine` 不带端口时，仍然可以等待远端 `Connect`。

2. （判断题）建链前需要完成两端内存注册，否则远端访问会失败。

3. （单选题）`Connect` 返回 `ALREADY_CONNECTED` 时应如何处理？
   A. 视为失败，立即退出
   B. 视为成功，继续传输
   C. 重新建链
   D. 报错后断链

4. （单选题）`Connect(remote_engine, timeout_in_millis)` 中的 `remote_engine` 应该来自哪里？
   A. 当前进程任意生成的字符串
   B. 远端 `Initialize` 使用的 `local_engine` 标识
   C. 本地 `MemHandle`
   D. 本地 Device ID

5. （多选题）`Connect` 的约束包括？
   A. `Initialize` 完成后才可调用
   B. 建链前完成内存注册
   C. 与 `Initialize` 同线程或通过 context 切换
   D. 建链后可以随时注册新内存

**执行以下代码获取答案。**


In [ ]:
!cat ./answer/02.04_answer.txt